# TerraSpectra — Day 8: Baseline 3D-CNN

**Task:** Implement a baseline 3D convolutional neural network for spatial-spectral hyperspectral patches and verify its forward pass.

**Input:** Day 7 PyTorch patch tensors with shape `(N, 1, 204, 9, 9)`.

**Important scientific limitation:** `pos_Plant21.csv` contains annotated coordinates, not verified healthy/diseased class labels. This notebook implements and tests the model architecture, but does not invent labels, train a disease classifier, or report disease-classification accuracy. The output layer has 3 configurable placeholder units, not asserted disease classes.

**Outputs:** Model definition, parameter report, forward-pass validation, initial (untrained) checkpoint, and reusable Python source.

## 1. Imports and device

In [ ]:
import os, json, zipfile
import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Upload the Day 7 ZIP or tensor files

In [ ]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded:
    print("Uploaded:", filename)

# Extract any uploaded ZIP files
for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as archive:
            archive.extractall("/content/day07")
        print("Extracted:", filename)

## 3. Find and load Day 7 patch tensors

In [ ]:
def find_file(filename):
    for root, _, filenames in os.walk("/content"):
        if filename in filenames:
            return os.path.join(root, filename)
    return None

paths = {
    "train": find_file("X_train_patches.pt"),
    "validation": find_file("X_validation_patches.pt"),
    "test": find_file("X_test_patches.pt")
}

if not all(paths.values()):
    raise FileNotFoundError(
        "Upload the Day 7 ZIP or X_train_patches.pt, "
        "X_validation_patches.pt, and X_test_patches.pt."
    )

X_train = torch.load(paths["train"], map_location="cpu", weights_only=True)
X_val = torch.load(paths["validation"], map_location="cpu", weights_only=True)
X_test = torch.load(paths["test"], map_location="cpu", weights_only=True)

print("Train:", tuple(X_train.shape))
print("Validation:", tuple(X_val.shape))
print("Test:", tuple(X_test.shape))

## 4. Validate input tensors

In [ ]:
expected_tail = (1, 204, 9, 9)
for name, tensor in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    assert tensor.ndim == 5
    assert tuple(tensor.shape[1:]) == expected_tail, (name, tensor.shape)
    assert tensor.dtype == torch.float32
    assert torch.isfinite(tensor).all()
print("All tensor dimensions and values are valid.")

## 5. Define the baseline 3D-CNN

In [ ]:
class TerraSpectra3DCNN(nn.Module):
    """3D-CNN for (N, 1, spectral_bands, height, width) inputs."""
    def __init__(self, num_classes=3, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),

            nn.Conv3d(8, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

NUM_OUTPUT_UNITS = 3  # Placeholder only; not verified disease classes.
model = TerraSpectra3DCNN(num_classes=NUM_OUTPUT_UNITS).to(device)
print(model)

## 6. Inspect model parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

## 7. Forward-pass smoke test using real Day 7 patches

In [ ]:
model.eval()
sample_batch = X_train[:min(4, len(X_train))].to(device)

with torch.no_grad():
    logits = model(sample_batch)
    feature_maps = model.features(sample_batch)

print("Input shape:", tuple(sample_batch.shape))
print("Feature extractor output:", tuple(feature_maps.shape))
print("Output logits shape:", tuple(logits.shape))
print("All outputs finite:", bool(torch.isfinite(logits).all()))

assert logits.shape == (sample_batch.shape[0], NUM_OUTPUT_UNITS)
assert tuple(feature_maps.shape[1:]) == (32, 1, 1, 1)
assert torch.isfinite(logits).all()
print("3D-CNN forward pass passed.")

## 8. Save model checkpoint and configuration

In [ ]:
output_dir = "/content/TerraSpectra_Day08"
os.makedirs(output_dir, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "model_name": "TerraSpectra3DCNN",
    "output_units_placeholder": NUM_OUTPUT_UNITS,
    "input_shape_per_sample": [1, 204, 9, 9],
    "status": "initialized_not_trained"
}, os.path.join(output_dir, "terraspectra_3dcnn_initial.pth"))

config = {
    "model_name": "TerraSpectra3DCNN",
    "input_layout": "(N, 1, spectral_bands, height, width)",
    "input_shape_per_sample": [1, 204, 9, 9],
    "convolution_channels": [1, 8, 16, 32],
    "output_units_placeholder": NUM_OUTPUT_UNITS,
    "dropout": 0.30,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "training_status": "not_trained",
    "reason": "Verified disease class labels are not available yet."
}
with open(os.path.join(output_dir, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)
print("Initial checkpoint and model configuration saved.")

## 9. Export reusable model source

In [ ]:
model_source = '''
import torch.nn as nn

class TerraSpectra3DCNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, 3, padding=1, bias=False),
            nn.BatchNorm3d(8), nn.ReLU(inplace=True),
            nn.MaxPool3d((2, 2, 2)),
            nn.Conv3d(8, 16, 3, padding=1, bias=False),
            nn.BatchNorm3d(16), nn.ReLU(inplace=True),
            nn.MaxPool3d((2, 2, 2)),
            nn.Conv3d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout), nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))
'''
with open(os.path.join(output_dir, "terraspectra_3dcnn.py"), "w") as f:
    f.write(model_source)
print("Reusable Python model file exported.")

## 10. Save implementation report

In [ ]:
report = {
    "day": 8,
    "task": "Baseline 3D-CNN architecture implementation",
    "train_samples": int(X_train.shape[0]),
    "validation_samples": int(X_val.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "input_shape_per_sample": list(X_train.shape[1:]),
    "output_units_placeholder": NUM_OUTPUT_UNITS,
    "total_parameters": int(total_params),
    "forward_pass": "passed",
    "model_trained": False,
    "classification_metrics": None,
    "label_status": "No verified disease class labels supplied"
}
with open(os.path.join(output_dir, "day08_implementation_report.json"), "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

## 11. Create downloadable output ZIP

In [ ]:
zip_path = "/content/TerraSpectra_Day08_Outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in os.listdir(output_dir):
        archive.write(
            os.path.join(output_dir, filename),
            arcname=os.path.join("TerraSpectra_Day08", filename)
        )

from google.colab import files
files.download(zip_path)

## Day 8 summary

- Implemented a baseline 3D-CNN for spatial-spectral hyperspectral patches.
- Inspected trainable parameters.
- Verified input, feature-map, and output tensor shapes with a forward pass.
- Saved an initial untrained checkpoint, configuration, and reusable model source.

**Important:** This is an architecture implementation and smoke test, not a trained disease classifier. Disease labels must be independently verified before supervised training and accuracy reporting.

**Suggested commit:**
`Day 8: Implement baseline 3D-CNN architecture for hyperspectral patches`

**Notebook path:**
`ml/notebooks/TerraSpectra_Day08_Baseline_3D_CNN.ipynb`